In [22]:
from pyspark.sql import types as T
from spark_session import get_spark
import sources

In [23]:
spark = get_spark("transform")

In [24]:
from schemas import ORDERS, CUSTOMERS, PRODUCTS
print(ORDERS)

StructType([StructField('order_id', LongType(), True), StructField('order_ts', TimestampType(), True), StructField('customer_id', IntegerType(), True), StructField('product_id', StringType(), True), StructField('quantity', IntegerType(), True), StructField('status', StringType(), True), StructField('updated_at', TimestampType(), True)])


In [25]:
orders =  sources.orders(spark)

In [26]:
orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [27]:
from pyspark.sql import functions as F

In [28]:
orders.select("order_id", "product_id", "quantity", "status").show(3)
paid = orders.filter((F.col("status") == "paid") & (F.col("quantity") > 0))
print("paid orders: ", paid.count())

print("bad quantity: ", orders.filter("quantity <= 0").count())

+--------+----------+--------+-------+
|order_id|product_id|quantity| status|
+--------+----------+--------+-------+
|  260001|      P002|       1|   paid|
|  260002|      P023|       3|   paid|
|  260003|      P016|       2|shipped|
+--------+----------+--------+-------+
only showing top 3 rows
paid orders:  208807
bad quantity:  2821


In [40]:
enriched_orders = (orders
                   .withColumn("order_date", F.to_date("order_ts"))
                   .withColumn("hour", F.hour("order_ts"))
                   .withColumn("size", F.when(F.col("quantity") >= 3, "large").otherwise("normal"))
                   .withColumn("quantity", F.col("quantity").cast("long"))
                   .withColumn("weekend", F.when((F.weekday("order_date") == 1) | (F.weekday("order_date") == 7), "TRUE").otherwise("FALSE"))
                   .withColumn("weekday", F.weekday(F.col("order_date")))
                   .withColumnRenamed("status", "order_status")
                   )
enriched_orders.select("order_id", "order_date", "hour", "quantity", "size", "order_status", "weekend", "weekday").show(5, truncate=False)

+--------+----------+----+--------+------+------------+-------+-------+
|order_id|order_date|hour|quantity|size  |order_status|weekend|weekday|
+--------+----------+----+--------+------+------------+-------+-------+
|260001  |2026-10-14|4   |1       |normal|paid        |FALSE  |2      |
|260002  |2026-10-14|4   |3       |large |paid        |FALSE  |2      |
|260003  |2026-10-14|6   |2       |normal|shipped     |FALSE  |2      |
|260004  |2026-10-14|12  |1       |normal|shipped     |FALSE  |2      |
|260005  |2026-10-14|18  |1       |normal|paid        |FALSE  |2      |
+--------+----------+----+--------+------+------------+-------+-------+
only showing top 5 rows


In [32]:
products = sources.products(spark)
(products
 .withColumn("code", F.concat_ws("-", F.upper(F.substring("category", 1, 3)), "product_id"))
 .withColumn("band", F.expr("CASE WHEN price < 50 THEN 'low' WHEN price < 200 THEN 'mid' ELSE 'high' END"))
 .select("code", "name", "price", "band")
 .show(5, truncate=False)
 )

+--------+------------------+------+----+
|code    |name              |price |band|
+--------+------------------+------+----+
|ELE-P001|Electronics item 1|61.51 |mid |
|HOM-P002|Home item 2       |84.79 |mid |
|BOO-P003|Books item 3      |46.3  |low |
|ELE-P004|Electronics item 4|547.82|high|
|ELE-P005|Electronics item 5|65.63 |mid |
+--------+------------------+------+----+
only showing top 5 rows


In [19]:
spark.stop()